***

* [总目录](../0_Introduction/0_introduction.ipynb)
* [术语表](../0_Introduction/1_glossary.ipynb)
* [7. 观测系统](7_0_introduction.ipynb)
    * 上一节： [第 7 章：观测系统](7_0_introduction.ipynb)
    * 下一节： [7.2 射电干涉测量方程](7_2_rime.ipynb)

***

导入标准模块:

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

plt.rcParams['figure.max_open_warning'] = 0

NOTEBOOK_DIR = Path('7_Observing_Systems') if Path('7_Observing_Systems').exists() else Path('.')
NOTEBOOK_DIR = NOTEBOOK_DIR.resolve()
STYLE_PATH = NOTEBOOK_DIR.parent / 'style' / 'course.css'
TOGGLE_PATH = NOTEBOOK_DIR.parent / 'style' / 'code_toggle.html'

try:
    from IPython.display import HTML
except ImportError:
    def HTML(*args, **kwargs):
        return None

if STYLE_PATH.exists():
    try:
        HTML(STYLE_PATH.read_text(encoding='utf-8'))
    except Exception:
        pass

本节以下示例全部使用 notebook 内生成的合成 Jones 向量、相干矩阵与随机电场样本，因此不依赖外部图片或专用软件包。与旧版的静态图相比，这一版更强调三件事：Jones 向量如何描述相干平面波，Stokes 参数为何本质上是统计量，以及 Jones 矩阵为什么必须按确定的次序组成链式模型。

In [ ]:
if TOGGLE_PATH.exists():
    try:
        HTML(TOGGLE_PATH.read_text(encoding='utf-8'))
    except Exception:
        pass

***

## 7.1 琼斯符号 <a id='instrum:sec:jones'></a>

琼斯记号（Jones notation 或 Jones calculus）是把电磁波传播与仪器响应写成线性代数问题的一套语言。它之所以重要，不是因为“2×2 矩阵看起来方便”，而是因为在窄带、准单色、线性系统的近似下，电场的两个横向分量恰好构成一个二维复向量，而绝大多数传播与接收效应都可以视为作用在这个二维向量上的线性变换。

因此，本节的核心任务有三步：

1. 先把相干平面波的极化状态写成 Jones 向量；
2. 再把非相干天体辐射写成相干矩阵或亮度矩阵；
3. 最后用 Jones 矩阵描述传播与仪器的线性作用，并说明为什么 Jones 链的次序不能随便交换。

后面的 7.2 节会把这些对象直接接到相关矩阵与 RIME；而 7.6 节则会进一步把 Jones/Mueller 语言应用到馈源和极化泄漏的具体硬件问题上。

In [ ]:
def normalize_jones(vec):
    vec = np.asarray(vec, dtype=complex).reshape(2)
    norm = np.sqrt(np.sum(np.abs(vec) ** 2))
    return vec if norm == 0 else vec / norm



def jones_linear(theta_deg=0.0):
    theta = np.deg2rad(theta_deg)
    return normalize_jones([np.cos(theta), np.sin(theta)])



def jones_circular(hand='R'):
    sign = -1j if hand.upper() == 'R' else 1j
    return normalize_jones([1.0, sign])



def jones_elliptical(amp_y=0.6, phase_deg=40.0):
    return normalize_jones([1.0, amp_y * np.exp(1j * np.deg2rad(phase_deg))])



def coherency_from_jones(jones, intensity=1.0):
    jones = normalize_jones(jones)
    return intensity * np.outer(jones, np.conjugate(jones))



def stokes_from_coherency(coh):
    xx = coh[0, 0]
    yy = coh[1, 1]
    xy = coh[0, 1]
    yx = coh[1, 0]
    I = np.real(xx + yy)
    Q = np.real(xx - yy)
    U = np.real(xy + yx)
    V = np.real(-1j * (xy - yx))
    return np.array([I, Q, U, V], dtype=float)



def coherency_from_stokes(stokes):
    I, Q, U, V = stokes
    return 0.5 * np.array(
        [[I + Q, U + 1j * V], [U - 1j * V, I - Q]],
        dtype=complex,
    )



def degree_of_polarization(stokes):
    I, Q, U, V = stokes
    return np.sqrt(Q ** 2 + U ** 2 + V ** 2) / max(I, 1e-12)



def trajectory(jones, n=400):
    phase = np.linspace(0.0, 2.0 * np.pi, n)
    ex = np.real(jones[0] * np.exp(1j * phase))
    ey = np.real(jones[1] * np.exp(1j * phase))
    return ex, ey



def jones_rotation(theta_deg):
    theta = np.deg2rad(theta_deg)
    c = np.cos(theta)
    s = np.sin(theta)
    return np.array([[c, s], [-s, c]], dtype=complex)



def jones_gain(gx=1.0, gy=1.0, phx_deg=0.0, phy_deg=0.0):
    return np.array(
        [
            [gx * np.exp(1j * np.deg2rad(phx_deg)), 0.0],
            [0.0, gy * np.exp(1j * np.deg2rad(phy_deg))],
        ],
        dtype=complex,
    )



def jones_leakage(dx=0.0, dy=0.0):
    return np.array([[1.0, dx], [dy, 1.0]], dtype=complex)



def apply_jones(jmat, jvec):
    return jmat @ jvec



def apply_jones_to_coherency(jmat, coh):
    return jmat @ coh @ np.conjugate(jmat.T)



def matrix_sqrt_psd(matrix):
    eigvals, eigvecs = np.linalg.eigh(matrix)
    eigvals = np.clip(eigvals, 0.0, None)
    return eigvecs @ np.diag(np.sqrt(eigvals)) @ np.conjugate(eigvecs.T)



def sample_field_from_coherency(coh, n=20000, seed=0):
    rng = np.random.default_rng(seed)
    sqrt_coh = matrix_sqrt_psd(coh)
    z = (rng.normal(size=(2, n)) + 1j * rng.normal(size=(2, n))) / np.sqrt(2.0)
    return sqrt_coh @ z



def estimate_coherency(samples):
    return (samples @ np.conjugate(samples.T)) / samples.shape[1]

### 7.1.1 相干平面波与 Jones 向量

对来自远处天体的窄带信号，我们通常把到达天线口径面的波前近似为平面波。若传播方向取为 $z$ 轴，则电场只有两个横向自由度，因此可写成

$$
\mathbf{e} = \begin{bmatrix} e_x \\ e_y \end{bmatrix}.
$$

这就是 Jones 向量。它是一个二维复向量，其中复数不仅记录振幅，也记录两个分量之间的相位差。正是这个相位差，决定了线极化、圆极化还是一般的椭圆极化。

需要特别注意的是：Jones 向量只适合描述**相干**的、相位关系确定的场。对于真正的天体辐射，我们稍后必须转向统计量。不过在进入统计描述之前，先把相干态看清楚会非常有帮助。

In [ ]:
examples = {
    'Linear X': jones_linear(0.0),
    'Linear 45 deg': jones_linear(45.0),
    'Right circular': jones_circular('R'),
    'Elliptical': jones_elliptical(0.55, 55.0),
}

print('相干 Jones 态与对应的 Stokes 参数：')
print(' state              Ex                     Ey                     I      Q      U      V')
for name, jvec in examples.items():
    stokes = stokes_from_coherency(coherency_from_jones(jvec))
    print(
        f' {name:<17s} '
        f'{jvec[0].real:+.3f}{jvec[0].imag:+.3f}j   '
        f'{jvec[1].real:+.3f}{jvec[1].imag:+.3f}j   '
        f'{stokes[0]:5.2f}  {stokes[1]:5.2f}  {stokes[2]:5.2f}  {stokes[3]:5.2f}'
    )

fig, axes = plt.subplots(1, 4, figsize=(15, 3.6))
for ax, (name, jvec) in zip(axes, examples.items()):
    ex, ey = trajectory(jvec)
    ax.plot(ex, ey, linewidth=2)
    ax.axhline(0.0, color='0.7', linewidth=0.8)
    ax.axvline(0.0, color='0.7', linewidth=0.8)
    ax.set_title(name)
    ax.set_xlabel('Re(Ex)')
    ax.set_ylabel('Re(Ey)')
    ax.set_aspect('equal')
    ax.grid(alpha=0.2)
plt.tight_layout()

从这些例子可以直接读出几个关键事实：

* 若两个分量同相或反相，轨迹退化成一条直线，对应线极化；
* 若两个分量振幅相等、相位差为 $\pm 90^\circ$，轨迹为圆，对应圆极化；
* 更一般地，轨迹为椭圆，对应椭圆极化。

同时也要注意，Jones 向量的整体复相位并没有独立物理意义；真正有物理意义的是两个分量之间的相对振幅和相对相位。不同文献在圆极化手性与 Stokes $V$ 的符号约定上可能不同，因此实际工作中一定要检查自己所采用的约定是否与软件和观测系统一致。

### 7.1.2 非相干辐射、Stokes 参数与亮度矩阵

真实天体源发出的辐射通常是非相干的。此时瞬时电场不再有稳定的确定相位关系，单个 Jones 向量不足以描述整个信号，必须转而使用二阶统计量，也就是相干矩阵

$$
\mathbf{C} = \langle \mathbf{e}\, \mathbf{e}^H \rangle.
$$

对二维横向电场而言，$\mathbf{C}$ 是一个 $2\times 2$ 埃尔米特矩阵。它与 Stokes 参数之间可双向转换：

$$
\mathbf{C} = \frac{1}{2}
\begin{bmatrix}
I+Q & U + iV \\
U-iV & I-Q
\end{bmatrix}.
$$

在 RIME 语境中，这个矩阵常写成亮度矩阵 $\mathbf{B}$。因此，亮度矩阵并不是“另一个新对象”，而正是用矩阵形式表达的 Stokes 信息。

In [ ]:
states = {
    'Fully linear X': np.array([1.0, 1.0, 0.0, 0.0]),
    'Partially linear': np.array([1.0, 0.70, 0.0, 0.0]),
    'Unpolarized': np.array([1.0, 0.0, 0.0, 0.0]),
}

print('不同统计极化态：')
print(' state               I      Q      U      V      p')
for name, stokes in states.items():
    pfrac = degree_of_polarization(stokes)
    print(f' {name:<18s} {stokes[0]:5.2f}  {stokes[1]:5.2f}  {stokes[2]:5.2f}  {stokes[3]:5.2f}  {pfrac:5.2f}')

exampleStokes = np.array([1.0, 0.25, -0.35, 0.15])
exampleBrightness = coherency_from_stokes(exampleStokes)
randomSamples = sample_field_from_coherency(exampleBrightness, n=30000, seed=4)
estimatedBrightness = estimate_coherency(randomSamples)
estimatedStokes = stokes_from_coherency(estimatedBrightness)

print()
print('目标 Stokes 与由随机电场样本回收的估计：')
print(' target   =', np.round(exampleStokes, 4))
print(' estimate =', np.round(estimatedStokes, 4))
print(' brightness matrix =')
print(np.round(exampleBrightness, 4))

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.5))

labels = ['Q/I', 'U/I', 'V/I']
x = np.arange(len(labels))
width = 0.23
for offset, (name, stokes) in zip([-width, 0.0, width], states.items()):
    axes[0].bar(x + offset, stokes[1:] / stokes[0], width=width, label=name)
axes[0].set_xticks(x)
axes[0].set_xticklabels(labels)
axes[0].set_ylim(-1.05, 1.05)
axes[0].set_title('Normalized polarization components')
axes[0].set_ylabel('Fraction of I')
axes[0].grid(axis='y', alpha=0.25)
axes[0].legend(fontsize=9)

axes[1].scatter(np.real(randomSamples[0, ::10]), np.real(randomSamples[1, ::10]), s=6, alpha=0.25)
axes[1].set_title('Instantaneous electric-field samples')
axes[1].set_xlabel('Re(Ex)')
axes[1].set_ylabel('Re(Ey)')
axes[1].axhline(0.0, color='0.7', linewidth=0.8)
axes[1].axvline(0.0, color='0.7', linewidth=0.8)
axes[1].set_aspect('equal')
axes[1].grid(alpha=0.2)

plt.tight_layout()

这里可以看到两层非常重要的区别：

* 对相干态，Jones 向量已经足够；
* 对非相干态，必须使用时间平均后的相干矩阵或 Stokes 参数。

这也是为什么真实天体辐射通常不用单个 Jones 向量直接表示，而要用 $I,Q,U,V$ 或亮度矩阵表示。与此同时，任何物理可实现的辐射都必须满足

$$
Q^2 + U^2 + V^2 \le I^2,
$$

也就是极化分数不能超过 100%。若在真实数据处理中出现形式上“超过 100%”的极化，往往意味着成像滤除了大尺度总强度结构、校准存在系统误差，或者不同 Stokes 量的空间频率覆盖并不一致。

### 7.1.3 Jones 矩阵：把传播和仪器写成线性变换

只要某个传播或接收过程对电场是线性的，它就可以写成一个 $2\times2$ 复矩阵：

$$
\mathbf{e}' = \mathbf{J}\,\mathbf{e}.
$$

这就是 Jones 矩阵。其物理意义非常直接：对角元通常代表两个正交极化通道各自的增益与相位，非对角元则描述通道间耦合或泄漏。

下面分别看三类最常见的 Jones 作用：基底旋转、两个通道之间的微分相位，以及少量的极化泄漏。

In [ ]:
inputJones = jones_linear(45.0)
inputStokes = stokes_from_coherency(coherency_from_jones(inputJones))

rotationJ = jones_rotation(30.0)
retarderJ = jones_gain(gx=1.0, gy=1.0, phx_deg=0.0, phy_deg=90.0)
rotationOut = apply_jones(rotationJ, inputJones)
retarderOut = apply_jones(retarderJ, inputJones)

unpolarizedCoh = coherency_from_stokes([1.0, 0.0, 0.0, 0.0])
leakageJ = jones_leakage(dx=0.10j, dy=-0.07j)
leakageOut = apply_jones_to_coherency(leakageJ, unpolarizedCoh)

print('Jones 矩阵作用前后的 Stokes：')
print(' input coherent state      =', np.round(inputStokes, 4))
print(' after rotation            =', np.round(stokes_from_coherency(coherency_from_jones(rotationOut)), 4))
print(' after differential phase  =', np.round(stokes_from_coherency(coherency_from_jones(retarderOut)), 4))
print(' unpolarized after leakage =', np.round(stokes_from_coherency(leakageOut), 4))

fig, axes = plt.subplots(1, 3, figsize=(13.5, 4))
for ax, (title, jvec) in zip(
    axes[:2],
    [
        ('Rotation of basis', rotationOut),
        ('Differential phase', retarderOut),
    ],
):
    ex, ey = trajectory(jvec)
    ax.plot(ex, ey, linewidth=2)
    ax.axhline(0.0, color='0.7', linewidth=0.8)
    ax.axvline(0.0, color='0.7', linewidth=0.8)
    ax.set_title(title)
    ax.set_xlabel('Re(Ex)')
    ax.set_ylabel('Re(Ey)')
    ax.set_aspect('equal')
    ax.grid(alpha=0.2)

leakStokes = stokes_from_coherency(leakageOut)
axes[2].bar(['I', 'Q', 'U', 'V'], leakStokes, color=['0.5', '#4c72b0', '#55a868', '#c44e52'])
axes[2].set_title('Leakage acting on unpolarized input')
axes[2].set_ylabel('Recovered Stokes')
axes[2].grid(axis='y', alpha=0.25)

plt.tight_layout()

这个结果对应三种非常常见的直觉：

* 旋转 Jones 项本质上是在改变极化基底，因此主要会在 $Q/U$ 之间重新分配信号；
* 微分相位项会在线极化与圆极化之间混合，因此一个线极化态经过四分之一波片式的相位延迟后可以转成圆极化；
* 泄漏项即使面对本来不偏振的输入，也可能在输出端制造出虚假的偏振分量，这正是极化校准中必须严肃处理的问题。

这些例子也解释了为什么 Jones 语言如此适合观测系统建模：每一种效应都可以先独立理解，再按实际信号路径串成链。

### 7.1.4 Jones 链与非交换性

一旦信号经过多个系统部件，Jones 矩阵就要按作用顺序相乘：

$$
\mathbf{J}_{\mathrm{tot}} = \mathbf{J}_n \cdots \mathbf{J}_2 \mathbf{J}_1.
$$

这一步看起来简单，但它有一个极其重要的后果：**Jones 矩阵通常不交换。** 也就是说，先旋转再加相位，与先加相位再旋转，通常不是同一个系统。RIME 之所以必须写成有序的 Jones 链，原因就在这里。

In [ ]:
sourceCoherency = coherency_from_jones(jones_linear(25.0), intensity=1.0)
rotationJ = jones_rotation(35.0)
retarderJ = jones_gain(gx=1.0, gy=1.0, phx_deg=0.0, phy_deg=90.0)
leakageJ = jones_leakage(dx=0.08j, dy=-0.05j)

chainA = leakageJ @ retarderJ @ rotationJ
chainB = rotationJ @ retarderJ @ leakageJ

outA = apply_jones_to_coherency(chainA, sourceCoherency)
outB = apply_jones_to_coherency(chainB, sourceCoherency)
stokesIn = stokes_from_coherency(sourceCoherency)
stokesA = stokes_from_coherency(outA)
stokesB = stokes_from_coherency(outB)

print('输入 Stokes                =', np.round(stokesIn, 4))
print('chain A output             =', np.round(stokesA, 4))
print('chain B output             =', np.round(stokesB, 4))
print('max |JA - JB|              =', np.max(np.abs(chainA - chainB)))
print('output Stokes difference   =', np.round(stokesA - stokesB, 4))

fig, ax = plt.subplots(figsize=(8.5, 4.5))
x = np.arange(4)
width = 0.24
ax.bar(x - width, stokesIn, width=width, label='Input')
ax.bar(x, stokesA, width=width, label='Chain A')
ax.bar(x + width, stokesB, width=width, label='Chain B')
ax.set_xticks(x)
ax.set_xticklabels(['I', 'Q', 'U', 'V'])
ax.set_ylabel('Stokes value')
ax.set_title('Order matters in a Jones chain')
ax.grid(axis='y', alpha=0.25)
ax.legend()
plt.tight_layout()

这一点会在下一节立刻变得关键：RIME 不是把若干“误差因子”随手乘起来，而是把信号从源到天线、再到相关器的作用顺序明确编码进一个有向链式模型。也正因为 Jones 项通常不交换，我们才能够在校准时区分出几何相位、电子增益、极化泄漏、主波束和传播介质等不同效应各自应当放在什么位置。

***

* 下一节： [7.2 射电干涉测量方程](7_2_rime.ipynb)